In [1]:
import tensorflow

print(tensorflow.__version__)

2.6.0


In [27]:
import glob #glob 모듈의 glob 함수는 사용자가 제시한 조건에 맞는 파일명을 리스트 형식으로 반환한다
import os, re
import numpy as np
import tensorflow as tf

txt_file_path = os.getenv('HOME')+'/aiffel/lyricist/data/lyrics/*' #os.getenv(x)함수는 환경 변수x의 값을 포함하는 문자열 변수를 반환합니다. txt_file_path 에 "/root/aiffel/lyricist/data/lyrics/*" 저장

txt_list = glob.glob(txt_file_path) #txt_file_path 경로에 있는 모든 파일명을 리스트 형식으로 txt_list 에 할당

raw_corpus = [] 

# 여러개의 txt 파일을 모두 읽어서 raw_corpus 에 담습니다.
for txt_file in txt_list:
    with open(txt_file, "r") as f:
        raw = f.read().splitlines() #read() : 파일 전체의 내용을 하나의 문자열로 읽어온다. , splitlines()  : 여러라인으로 구분되어 있는 문자열을 한라인씩 분리하여 리스트로 반환
        raw_corpus.extend(raw) # extend() : 리스트함수로 추가적인 내용을 연장 한다.

print("데이터 크기:", len(raw_corpus))
print("Examples:\n", raw_corpus[:3])

데이터 크기: 187088
Examples:
 ['The first words that come out', 'And I can see this song will be about you', "I can't believe that I can breathe without you"]


### 데이터 정제

In [28]:
# 입력된 문장을
#     1. 소문자로 바꾸고, 양쪽 공백을 지웁니다
#     2. 특수문자 양쪽에 공백을 넣고
#     3. 여러개의 공백은 하나의 공백으로 바꿉니다
#     4. a-zA-Z?.!,¿가 아닌 모든 문자를 하나의 공백으로 바꿉니다
#     5. 다시 양쪽 공백을 지웁니다
#     6. 문장 시작에는 <start>, 끝에는 <end>를 추가합니다
# 이 순서로 처리해주면 문제가 되는 상황을 방지할 수 있겠네요!
def preprocess_sentence(sentence):
    sentence = sentence.lower().strip() # 1
    sentence = re.sub(r"([?.!,¿])", r" \1 ", sentence) # 2
    sentence = re.sub(r'[" "]+', " ", sentence) # 3
    sentence = re.sub(r"[^a-zA-Z?.!,¿]+", " ", sentence) # 4
    sentence = sentence.strip() # 5
    sentence = '<start> ' + sentence + ' <end>' # 6
    return sentence

# 이 문장이 어떻게 필터링되는지 확인해 보세요.
print(preprocess_sentence("This @_is ;;;sample        sentence."))

<start> this is sample sentence . <end>


In [29]:
# 여기에 정제된 문장을 모을겁니다
corpus = []

# raw_corpus list에 저장된 문장들을 순서대로 반환하여 sentence에 저장
for sentence in raw_corpus:
    preprocessed_sentence = preprocess_sentence(sentence)
    corpus.append(preprocessed_sentence)
        
# 정제된 결과를 10개만 확인해보죠
corpus[:10]

['<start> the first words that come out <end>',
 '<start> and i can see this song will be about you <end>',
 '<start> i can t believe that i can breathe without you <end>',
 '<start> but all i need to do is carry on <end>',
 '<start> the next line i write down <end>',
 '<start> and there s a tear that falls between the pages <end>',
 '<start> i know that pain s supposed to heal in stages <end>',
 '<start> but it depends which one i m standing on i write lines down , then rip them up <end>',
 '<start> describing love can t be this tough i could set this song on fire , send it up in smoke <end>',
 '<start> i could throw it in the river and watch it sink in slowly <end>']

### 평가 데이터셋 분리

In [30]:

def tokenize(corpus):
    tokenizer = tf.keras.preprocessing.text.Tokenizer(
        num_words=12000, 
        filters=' ',
        oov_token="<unk>"
    )
    # corpus를 이용해 tokenizer 내부의 단어장을 완성합니다
    # tokenizer.fit_on_texts(texts): 문자 데이터를 입력받아 리스트의 형태로 변환하는 메서드
    tokenizer.fit_on_texts(corpus)
    # 준비한 tokenizer를 이용해 corpus를 Tensor로 변환합니다
    # tokenizer.texts_to_sequences(texts): 텍스트 안의 단어들을 숫자의 시퀀스 형태로 변환하는 메서드
    tensor = tokenizer.texts_to_sequences(corpus)   
    # 입력 데이터의 시퀀스 길이를 일정하게 맞춰줍니다
    # 만약 시퀀스가 짧다면 문장 뒤에 패딩을 붙여 길이를 맞춰줍니다.
    # 문장 앞에 패딩을 붙여 길이를 맞추고 싶다면 padding='pre'를 사용합니다
    tensor = tf.keras.preprocessing.sequence.pad_sequences(tensor, padding='post',maxlen=15)  
    
    print(tensor,tokenizer)
    return tensor, tokenizer

tensor, tokenizer = tokenize(corpus)

[[  2   6 241 ...   0   0   0]
 [  2   8   5 ...   0   0   0]
 [  2   5  32 ...   0   0   0]
 ...
 [  2  39  39 ...   0   0   0]
 [  2   5  22 ...   0   0   0]
 [  2  39  39 ...   0   0   0]] <keras_preprocessing.text.Tokenizer object at 0x7dd448366c40>


In [31]:
# tokenizer.index_word: 현재 계산된 단어의 인덱스와 인덱스에 해당하는 단어를 dictionary 형대로 반환 (Ex. {index: '~~', index: '~~', ...})
for idx in tokenizer.index_word:
    print(idx, ":", tokenizer.index_word[idx])

    if idx >= 10: break

1 : <unk>
2 : <start>
3 : <end>
4 : ,
5 : i
6 : the
7 : you
8 : and
9 : a
10 : to


In [32]:
from sklearn.model_selection import train_test_split

enc_input = tensor[:, :-1]  
dec_input = tensor[:, 1:]    

enc_train, enc_val, dec_train, dec_val = train_test_split(
    enc_input, 
    dec_input, 
    test_size=0.2,
    random_state=42
    )

### 인공지능 만들기

In [33]:
BUFFER_SIZE = len(enc_input)
BATCH_SIZE = 64
steps_per_epoch = len(enc_input) // BATCH_SIZE

 
VOCAB_SIZE = tokenizer.num_words + 1   


dataset = tf.data.Dataset.from_tensor_slices((enc_input, dec_input))
dataset = dataset.shuffle(BUFFER_SIZE)
dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)
print(dataset)

val_dataset = tf.data.Dataset.from_tensor_slices((enc_val, dec_val))
val_dataset = val_dataset.shuffle(BUFFER_SIZE)
val_dataset = val_dataset.batch(BATCH_SIZE, drop_remainder=True)
print(val_dataset)

<BatchDataset shapes: ((64, 14), (64, 14)), types: (tf.int32, tf.int32)>
<BatchDataset shapes: ((64, 14), (64, 14)), types: (tf.int32, tf.int32)>


In [34]:
class TextGenerator(tf.keras.Model):
    def __init__(self, vocab_size, embedding_size, hidden_size):
        super().__init__()
    
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_size) 
        self.rnn_1 = tf.keras.layers.LSTM(hidden_size, return_sequences=True)  
        self.rnn_2 = tf.keras.layers.LSTM(hidden_size, return_sequences=True)
        self.linear = tf.keras.layers.Dense(vocab_size)
        
    def call(self, x):
        out = self.embedding(x)
        out = self.rnn_1(out)
        out = self.rnn_2(out)
        out = self.linear(out)
        
        return out

embedding_size = 256 # 워드 벡터의 차원수를 말하며 단어가 추상적으로 표현되는 크기입니다.
hidden_size = 1024 # 모델에 얼마나 많은 일꾼을 둘 것인가? 정도로 이해하면 좋다.
model = TextGenerator(tokenizer.num_words + 1, embedding_size , hidden_size) # tokenizer.num_words에 +1인 이유는 문장에 없는 pad가 사용되었기 때문이다.

In [35]:
#Loss
# tf.keras.losses.SparseCategoricalCrossentropy : https://www.tensorflow.org/api_docs/python/tf/keras/losses/SparseCategoricalCrossentropy
optimizer = tf.keras.optimizers.Adam()

loss = tf.keras.losses.SparseCategoricalCrossentropy( 
    from_logits=True, reduction='none') # 클래스 분류 문제에서 softmax 함수를 거치면 from_logits = False(default값),그렇지 않으면 from_logits = True.

model.compile(loss=loss, optimizer=optimizer)
model.fit(dataset, validation_data = val_dataset, epochs=10)

Epoch 1/10
2923/2923 [==============================] - 161s 54ms/step - loss: 3.1276 - val_loss: 2.7828
Epoch 2/10
2923/2923 [==============================] - 157s 54ms/step - loss: 2.6908 - val_loss: 2.4910
Epoch 3/10
2923/2923 [==============================] - 157s 54ms/step - loss: 2.4397 - val_loss: 2.2481
Epoch 4/10
2923/2923 [==============================] - 157s 54ms/step - loss: 2.2182 - val_loss: 2.0310
Epoch 5/10
2923/2923 [==============================] - 157s 54ms/step - loss: 2.0271 - val_loss: 1.8432
Epoch 6/10
2923/2923 [==============================] - 157s 54ms/step - loss: 1.8583 - val_loss: 1.6849
Epoch 7/10
2923/2923 [==============================] - 157s 54ms/step - loss: 1.7087 - val_loss: 1.5414
Epoch 8/10
2923/2923 [==============================] - 157s 54ms/step - loss: 1.5750 - val_loss: 1.4113
Epoch 9/10
2923/2923 [==============================] - 157s 54ms/step - loss: 1.4573 - val_loss: 1.3057
Epoch 10/10
2923/2923 [==============================] 

In [133]:
#문장생성 함수 정의
#모델에게 시작 문장을 전달하면 모델이 시작 문장을 바탕으로 작문을 진행
def generate_text(model, tokenizer, init_sentence="<start>", max_len=20): #시작 문자열을 init_sentence 로 받으며 디폴트값은 <start> 를 받는다
    # 테스트를 위해서 입력받은 init_sentence도 텐서로 변환합니다
    test_input = tokenizer.texts_to_sequences([init_sentence]) #텍스트 안의 단어들을 숫자의 시퀀스의 형태로 변환
    test_tensor = tf.convert_to_tensor(test_input, dtype=tf.int64)
    end_token = tokenizer.word_index["<end>"]

    # 단어 하나씩 예측해 문장을 만듭니다
    #    1. 입력받은 문장의 텐서를 입력합니다
    #    2. 예측된 값 중 가장 높은 확률인 word index를 뽑아냅니다
    #    3. 2에서 예측된 word index를 문장 뒤에 붙입니다
    #    4. 모델이 <end>를 예측했거나, max_len에 도달했다면 문장 생성을 마칩니다 (도달 하지 못하였으면 while 루프를 돌면서 다음 단어를 예측)
    while True: #루프를 돌면서 init_sentence에 단어를 하나씩 생성성
        # 1
        predict = model(test_tensor) 
        # 2
        predict_word = tf.argmax(tf.nn.softmax(predict, axis=-1), axis=-1)[:, -1] 
        # 3 
        test_tensor = tf.concat([test_tensor, tf.expand_dims(predict_word, axis=0)], axis=-1)
        # 4 
        if predict_word.numpy()[0] == end_token: break
        if test_tensor.shape[1] >= max_len: break

    generated = ""
    # tokenizer를 이용해 word index를 단어로 하나씩 변환합니다 
    for word_index in test_tensor[0].numpy():
        if word_index == 0:
            continue
        generated += tokenizer.index_word[word_index] + " "
    return generated

In [134]:
generate_text(model, tokenizer, init_sentence="<start> i love", max_len=20)
# generate_text 함수에 lyricist 라 정의한 모델을 이용해서 ilove 로 시작되는 문장을 생성

'<start> i love you more than i want to <end> '

In [127]:
import numpy as np

def sample_next_word(preds, temperature=0.6, top_k=7):

    preds = np.asarray(preds).astype("float64")

    # softmax 확률 안정화
    preds = np.clip(preds, 1e-10, 1)

    preds = np.log(preds) / temperature
    preds = np.exp(preds)
    preds = preds / np.sum(preds)

    # top-k 후보만 선택
    top_indices = preds.argsort()[-top_k:]
    top_probs = preds[top_indices]

    top_probs = top_probs / np.sum(top_probs)

    return np.random.choice(top_indices, p=top_probs)


In [102]:
import tensorflow as tf

def generate_text(model, tokenizer, init_sentence="<start>", max_len=20):

    sentence = init_sentence
    words = sentence.split()

    for _ in range(max_len):

        encoded = tokenizer.texts_to_sequences([sentence])
        encoded = tf.keras.preprocessing.sequence.pad_sequences(
            encoded, maxlen=max_len-1, padding='pre'
        )

        preds = model.predict(encoded, verbose=0)[0][-1]

        # logits → probability
        preds = tf.nn.softmax(preds).numpy()

        next_index = sample_next_word(preds)

        if next_index == 0:
            continue

        next_word = tokenizer.index_word.get(next_index, "")

        if next_word == "<end>":
            break

        # 같은 단어 반복 방지
        if len(words) >= 2 and words[-1] == next_word and words[-2] == next_word:
            continue

        words.append(next_word)
        sentence += " " + next_word

    return " ".join(words) + " <end>"


In [124]:
def generate_song(model, tokenizer, init_sentence, num_lines=5, max_len=20):

    sentence = init_sentence
    generated_set = set()

    for _ in range(num_lines):

        for _ in range(10):  # 중복 방지 재시도
            generated = generate_text(model, tokenizer, sentence, max_len)

            if generated not in generated_set:
                generated_set.add(generated)
                print(generated)
                break

        words = [w for w in generated.split() if w not in ["<start>", "<end>"]]

        sentence = "<start> " + " ".join(words[-3:])




In [154]:
generate_song(model, tokenizer, "<start> i like", num_lines=5)


<start> i like the way how you re lovin me <end> 
<start> re lovin me to be alright <end> 
<start> to be alright oh oh oh <end> 
<start> oh oh oh oh oh oh oh oh <end> 
